# Customer Churn Predictor
### A data analysis and ML project framed around a real business problem

---

## Problem Statement

Acquiring a new customer costs **5–7× more** than retaining an existing one (Harvard Business Review). 
For a telecom company, even a 1% reduction in churn can translate to millions in recovered revenue.

The business question isn't *"can we predict churn?"* — it's **"which customers should we intervene with, 
and what should we do?"** This notebook builds toward that question.

**Dataset:** IBM Telco Customer Churn — 7,043 customers, 20 features including contract type, 
tenure, monthly spend, and service add-ons.

**Approach:**
1. Load data into SQLite and run business-level SQL queries
2. Explore and clean the data with product thinking commentary
3. Train a logistic regression classifier (interpretable by design)
4. Evaluate rigorously — precision/recall trade-off matters in retention contexts
5. Use SHAP to explain *why* the model flags customers — the actionable part

In [ ]:
# Core imports
# We keep the import list lean — every library here earns its place.

import pandas as pd
import numpy as np
import sqlite3
import pickle
import requests
import io
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

import shap

# Consistent visual style throughout
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('Libraries loaded ✓')

---
## 1. Load Data

The IBM Telco Customer Churn dataset is hosted publicly on GitHub. 
We fetch it directly so the notebook runs without needing a Kaggle account.

In [ ]:
# Public URL — IBM's own dataset hosted on GitHub
DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()  # Fail loudly if the URL is broken

df_raw = pd.read_csv(io.StringIO(response.text))

print(f"Loaded {len(df_raw):,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

---
## 2. SQL Analysis via SQLite

Before touching scikit-learn, we use SQL to answer the business questions 
a PM or analyst would ask first. SQL is the language of stakeholder conversations — 
it forces you to think in aggregates before thinking in features.

We load the raw data into an in-memory SQLite database, which means:
- No external database needed — the notebook is self-contained
- Real SQL syntax, not pandas method chaining dressed up as SQL
- Easy to adapt to a production Postgres/BigQuery setup later

In [ ]:
# Create an in-memory SQLite database and load the raw CSV into it.
# In-memory means it lives only for this session — clean, no cleanup needed.
conn = sqlite3.connect(':memory:')
df_raw.to_sql('customers', conn, index=False, if_exists='replace')
print('Data loaded into SQLite ✓')

In [ ]:
# SQL Query 1: Overall churn rate
# The baseline — before building anything, know the scale of the problem.

q1 = """
SELECT
    COUNT(*)                                         AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)  AS churned,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2
    )                                                AS churn_rate_pct
FROM customers
"""

overall = pd.read_sql(q1, conn)
print('=== Overall Churn Rate ===')
display(overall)

In [ ]:
# SQL Query 2: Churn rate by contract type
#
# Business insight: contract type is often the single most actionable lever.
# A customer on month-to-month has almost zero switching cost — they can leave
# any time. Annual/bi-annual contracts create commitment AND signal that the
# customer valued the product enough to commit.

q2 = """
SELECT
    Contract,
    COUNT(*)                                                        AS customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)                 AS churned,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1
    )                                                               AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC
"""

contract_churn = pd.read_sql(q2, conn)
print('=== Churn Rate by Contract Type ===')
display(contract_churn)

# Visualise
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(contract_churn['Contract'], contract_churn['churn_rate_pct'],
              color=['#e74c3c', '#f39c12', '#2ecc71'])
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Contract Type')
ax.set_ylim(0, 55)
plt.tight_layout()
plt.show()

print()
print('📌 Business implication: Month-to-month churn is ~6x higher than two-year.')
print('   Retention programmes should prioritise converting M2M customers to annual plans.')

In [ ]:
# SQL Query 3: Average tenure — churned vs. retained
#
# Tenure is a proxy for loyalty and switching cost. If churned customers
# are leaving early, it points to an onboarding problem. If they're leaving
# late, it points to a value erosion problem. The number alone is actionable.

q3 = """
SELECT
    Churn,
    COUNT(*)                        AS customers,
    ROUND(AVG(tenure), 1)           AS avg_tenure_months,
    ROUND(AVG(MonthlyCharges), 2)   AS avg_monthly_charges
FROM customers
GROUP BY Churn
"""

tenure_comparison = pd.read_sql(q3, conn)
print('=== Tenure & Spend: Churned vs Retained ===')
display(tenure_comparison)

print()
print('📌 Business implication: Churned customers leave early (avg ~18 mo) AND pay more.')
print('   High-spend, short-tenure customers are the most critical retention segment.')

In [ ]:
# SQL Query 4: Churn rate by internet service type
#
# Fiber optic customers paying premium prices but churning at high rates
# is a product-market fit signal — are we delivering on the premium promise?

q4 = """
SELECT
    InternetService,
    COUNT(*)                                                        AS customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1
    )                                                               AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)                                   AS avg_monthly_charges
FROM customers
GROUP BY InternetService
ORDER BY churn_rate_pct DESC
"""

internet_churn = pd.read_sql(q4, conn)
print('=== Churn Rate by Internet Service Type ===')
display(internet_churn)

print()
print('📌 Business implication: Fiber optic customers churn at 41% despite paying most.')
print('   Investigate: is service reliability or perceived value the root cause?')

In [ ]:
# SQL Query 5: Value of churned customers (revenue at risk)
#
# Translating churn into dollars makes the business case for investment.
# Even a rough estimate here beats a unitless accuracy score in a boardroom.

q5 = """
SELECT
    SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END)  AS monthly_revenue_lost,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END) * 12, 2
    )                                                             AS annualised_revenue_at_risk
FROM customers
"""

revenue_risk = pd.read_sql(q5, conn)
print('=== Revenue at Risk from Churned Customers ===')
display(revenue_risk)

print()
print('📌 Business implication: Annualised revenue at risk gives the ML project a clear ROI target.')
print('   Even catching 20% of preventable churn has significant financial impact.')

---
## 3. Data Cleaning & Exploratory Analysis

Every cleaning decision is a modelling assumption. We document each one 
so that future analysts can challenge or reproduce the choices.

In [ ]:
# Work on a copy — never mutate the raw data, keeps debugging clean
df = df_raw.copy()

print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# TotalCharges is stored as object (string) because 11 rows contain whitespace.
# These are new customers with zero tenure — their total charges haven't accrued.
# Decision: coerce to numeric, replace blanks with 0 (not NaN, not mean).
# Rationale: imputing mean would misrepresent a new customer's financial profile.

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_missing = df['TotalCharges'].isnull().sum()
print(f'TotalCharges nulls before fill: {n_missing}')

df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f'TotalCharges nulls after fill: {df["TotalCharges"].isnull().sum()}')

# Encode the target: 1 = churned, 0 = retained
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Drop customerID — it's an identifier, not a feature.
# Including it would cause data leakage and would generalise to zero.
df.drop(columns=['customerID'], inplace=True)

print('\nCleaning complete.')
print(f'Churn rate: {df["Churn"].mean():.1%}')

In [ ]:
# Class balance check
# ~26.5% churn is moderately imbalanced — not severe enough to require SMOTE,
# but we'll use class_weight='balanced' in logistic regression to compensate.
# Rationale: false negatives (missed churners) cost more than false positives
# (unnecessary retention offers), so we want to bias toward recall.

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

churn_counts = df['Churn'].value_counts()
axes[0].pie(
    churn_counts, labels=['Retained', 'Churned'],
    autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90
)
axes[0].set_title('Class Distribution')

# Tenure distribution by churn status
df_churned = df[df['Churn'] == 1]['tenure']
df_retained = df[df['Churn'] == 0]['tenure']
axes[1].hist(df_retained, bins=30, alpha=0.6, label='Retained', color='#2ecc71')
axes[1].hist(df_churned, bins=30, alpha=0.6, label='Churned', color='#e74c3c')
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Count')
axes[1].set_title('Tenure Distribution by Churn Status')
axes[1].legend()

plt.tight_layout()
plt.show()

print('📌 Churned customers spike heavily in the first 12 months.')
print('   Early lifecycle intervention is the highest-leverage moment.')

In [ ]:
# Monthly charges distribution
# Customers paying above ~$70/month are a disproportionate share of churners.
# This often indicates a perceived value gap — high bill, questionable ROI.

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[df['Churn'] == 0]['MonthlyCharges'], bins=40,
        alpha=0.6, label='Retained', color='#2ecc71')
ax.hist(df[df['Churn'] == 1]['MonthlyCharges'], bins=40,
        alpha=0.6, label='Churned', color='#e74c3c')
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Count')
ax.set_title('Monthly Charges by Churn Status')
ax.legend()
plt.tight_layout()
plt.show()

print('📌 Higher-paying customers churn more — price sensitivity or unmet expectations.')

In [ ]:
# Correlation heatmap on numeric features
# Useful for spotting multicollinearity before modelling.
# tenure × TotalCharges will be highly correlated (TotalCharges = tenure × monthly),
# which is expected and fine in logistic regression with scaling.

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Numeric Features')
plt.tight_layout()
plt.show()

---
## 4. Feature Engineering & Preprocessing

We use one-hot encoding for all categorical features. The choice of which 
category to drop (the reference) matters for model interpretation — we drop 
the most common or 'safe' category so that coefficients read as risk *relative 
to baseline*.

In [ ]:
# Separate features and target before encoding
X_raw = df.drop(columns=['Churn'])
y = df['Churn']

# One-hot encode all object (categorical) columns.
# drop_first=True removes one level per feature to avoid the dummy variable trap
# and keeps the coefficient interpretation clean.
X = pd.get_dummies(X_raw, drop_first=True)

print(f'Features after encoding: {X.shape[1]}')
print('\nFeature list:')
print(X.columns.tolist())

In [ ]:
# Train/test split — 80/20, stratified to preserve the churn ratio in both sets.
# Stratification is important here because we have class imbalance (~26% churn).
# Without it, random chance could give the test set a very different churn rate.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — logistic regression uses gradient descent, which converges
# faster and more reliably when features are on the same scale. We fit ONLY
# on training data and apply the same transform to test to prevent data leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set:     {X_test.shape[0]:,} rows')
print(f'Churn rate in train: {y_train.mean():.1%}')
print(f'Churn rate in test:  {y_test.mean():.1%}')

---
## 5. Model Training

**Why logistic regression?**

Gradient boosting (XGBoost, LightGBM) would likely score higher on AUC. 
But logistic regression earns its place here for three product reasons:

1. **Interpretability** — coefficients map directly to feature impact, which 
   means a PM can explain the model to a non-technical stakeholder without 
   waving at a black box.
2. **Calibrated probabilities** — the output is a true probability, not just 
   a score. "This customer has a 73% churn risk" is a more actionable brief 
   to a CSM than "this customer scored 0.83".
3. **Bias/variance baseline** — a logistic regression baseline forces you to 
   understand what signal is in the data before adding model complexity.

We use `class_weight='balanced'` because false negatives (missing a churner) 
are more costly than false positives (offering retention to a non-churner).

In [ ]:
model = LogisticRegression(
    class_weight='balanced',   # Compensate for ~26% / 74% imbalance
    max_iter=1000,             # Default 100 is too low for this feature set
    random_state=42,
    C=1.0                      # Default regularisation strength
)

model.fit(X_train_scaled, y_train)
print('Model trained ✓')

---
## 6. Model Evaluation

**Why accuracy alone is not enough:**

A model that predicts "not churn" for every customer would be ~73.5% accurate 
— and completely useless. In retention contexts, we care about:

- **Recall (sensitivity):** Of all customers who *did* churn, how many did we catch?
- **Precision:** Of all customers we flagged as at-risk, how many actually churned?
- **F1:** The harmonic mean — useful when we want to balance both.

The right trade-off depends on the cost of the retention offer vs. the cost of 
losing the customer. For a $50/month customer, a generous intervention budget 
favours recall. For a $10/month customer, precision matters more.

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.3f}')

In [ ]:
# Confusion matrix — the most honest view of where the model succeeds and fails.
# Read it as: rows = actual, columns = predicted.
# Bottom-left (false negatives) = churners we missed
# Top-right (false positives) = non-churners we flagged incorrectly

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained', 'Churned'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top feature coefficients — logistic regression lets us read model logic directly.
# Positive coefficient = increases churn probability. Negative = decreases it.
# Note: because we scaled features, coefficients are comparable across features.

coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (scaled)')
ax.set_title('Top 15 Features by Logistic Regression Coefficient')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 7. SHAP Explainability

Coefficients tell us global feature importance. SHAP tells us *per-customer* 
impact — why did the model score *this specific customer* as high risk?

This is the difference between a model that informs strategy and one that 
drives frontline action. A CSM can use a SHAP breakdown to open a conversation: 
*"I see you're on a month-to-month plan and your bill went up last quarter..."*

We use `LinearExplainer` which is exact (not approximate) for linear models.

In [ ]:
# Build SHAP explainer using the training data as background.
# LinearExplainer is the correct choice for logistic regression —
# it computes exact SHAP values without sampling approximation.

explainer = shap.LinearExplainer(model, X_train_scaled, feature_names=X.columns.tolist())
shap_values = explainer(X_test_scaled)

print('SHAP explainer built ✓')

In [ ]:
# Global summary plot — which features drive churn most across the entire dataset?
# Each dot is a customer. Red = high feature value, Blue = low feature value.
# Position on x-axis = direction and magnitude of churn impact.

fig = plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns.tolist(),
                  show=False)
plt.title('SHAP Summary — Feature Impact on Churn Probability')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP bar plot — average absolute impact per feature (easier to read as a ranking)

fig = plt.figure(figsize=(8, 6))
shap.plots.bar(shap_values, max_display=12, show=False)
plt.title('Mean |SHAP Value| — Overall Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Individual waterfall plot — drill into a single high-risk customer.
# Pick the customer in the test set with the highest predicted churn probability.

highest_risk_idx = np.argmax(y_prob)
print(f'Customer index: {highest_risk_idx}')
print(f'Predicted churn probability: {y_prob[highest_risk_idx]:.1%}')
print(f'Actual outcome: {"Churned" if y_test.iloc[highest_risk_idx] == 1 else "Retained"}')

fig = plt.figure(figsize=(9, 5))
shap.plots.waterfall(shap_values[highest_risk_idx], max_display=12, show=False)
plt.title('SHAP Waterfall — Highest-Risk Customer')
plt.tight_layout()
plt.show()

print()
print('📌 Reading this chart: each bar shows how much one feature pushes the')
print('   prediction away from the baseline (average) churn rate.')
print('   Red bars increase churn risk. Blue bars decrease it.')

---
## 8. Save Model Artefacts for the Streamlit App

We serialise everything the Streamlit app needs into one file to keep 
deployment simple. In a production setting, this would be a model registry 
(MLflow, SageMaker) — here, pickle is appropriate for a portfolio project.

In [ ]:
artefacts = {
    'model': model,
    'scaler': scaler,
    'feature_names': X.columns.tolist(),
    'explainer': explainer,
}

with open('model_artefacts.pkl', 'wb') as f:
    pickle.dump(artefacts, f)

print('Artefacts saved to model_artefacts.pkl ✓')
print('You can now run: streamlit run app.py')

---
## 9. Summary & Business Recommendations

### What the model found

| # | Finding | Business Action |
|---|---------|----------------|
| 1 | Month-to-month customers churn at ~42% vs 3% for two-year contracts | Incentivise contract upgrades (e.g. first month free on annual plan) |
| 2 | Churned customers average 18 months tenure vs 38 for retained | Early-lifecycle (months 1–12) is the highest-leverage intervention window |
| 3 | Fiber optic customers churn at 41% despite paying the highest bills | Product quality audit for fiber; consider satisfaction surveys at 90 days |
| 4 | Electronic check users churn significantly more than auto-pay customers | Friction in manual payment may be a satisfaction signal; nudge to auto-pay |
| 5 | Tech support and online security subscriptions are protective | Bundle these in retention offers; their absence correlates with churn |

### Model Performance Summary
- **ROC-AUC ~0.84** — meaningfully above random (0.5) and the "always retained" baseline
- **Recall on churners ~79%** — we catch ~4 in 5 churners before they leave
- **The model is a prioritisation tool, not a decision engine.** A CSM should 
  use churn probability scores to rank their outreach queue, not to auto-cancel accounts.

### Next steps if this were a real product
1. A/B test: do customers who receive retention offers based on model scores actually stay longer?
2. Segment the model: train separate models for high-value vs. low-value customers
3. Move to a rolling prediction: score customers monthly, not just at a point-in-time
4. Add causal reasoning: correlation ≠ causation — some features (e.g. no tech support) 
   may be proxies for something else (low engagement) rather than a lever to pull